### Reranking

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the queryIt ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [2]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser 

d:\project\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load text file
loader=TextLoader("data/langchain_sample.txt")
raw_docs=loader.load()

#Split text into document chunks
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
docs=splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'data/langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.'),
 Docu

In [4]:
#User query

query="How can I use langchain to build an application with memory and tools?"

In [5]:
#FAISS and HuggingFace Embeddings

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embedding_model)
retriever=vectorstore.as_retriever(search_kwargs={"k":8})

In [ ]:
#GROQ AI Embeddings
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"]

In [7]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [8]:
#Reranking - prompt and use LLM
llm=init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002376EB54440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002376EB55160>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
#Prompt Template for reranking

prompt=PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant.
                                    
User Question: {question}
                                    
Documents: {documents}
                                    
Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.
            
Output Format: comma seperated document indices (e.g., 2,1,3,0...)
""")

In [10]:
retrieved_docs=retriever.invoke(query)
retrieved_docs

[Document(id='4b144157-1196-48f9-a110-34799a2f6194', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='172ed8c6-0574-486d-be79-8cfbc039c616', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='fc6abe0f-5208-4a11-8b6f-d06ee1c3080a', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems

In [11]:
chain=prompt | llm | StrOutputParser
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant.\n\nUser Question: {question}\n\nDocuments: {documents}\n\nInstructions:\n- Think about the relevance of each document to the user's question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput Format: comma seperated document indices (e.g., 2,1,3,0...)\n")
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002376EB54440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002376EB55160>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))
| RunnableLambda(StrOutputParser)

In [12]:
#Combine all the retrieved docs
doc_lines=[f"{i+1},{doc.page_content}" for i,doc in enumerate(retrieved_docs)]
formatted_docs="\n".join(doc_lines)

In [13]:
resp=chain.invoke({"question":query,"documents":formatted_docs})

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given